# MiniAI — Обучение мини-модели

1. Включите GPU: Runtime → Change runtime type → T4 GPU
2. Запускайте ячейки по порядку

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/YOUR_USERNAME/miniai-project.git
%cd miniai-project

# ИЛИ загрузите файлы вручную через Files → Upload
# Затем раскомментируйте:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/project

In [ ]:
!pip install torch numpy gradio -q

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
!python data/prepare.py

In [ ]:
!python training/train.py

In [ ]:
# Тест inference
import sys, os
sys.path.insert(0, '.')
import torch
from model.config import PROJECT_ROOT, InferenceConfig
from inference.chat import load_model, generate_response

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tokenizer, mc = load_model(
    os.path.join(PROJECT_ROOT, 'checkpoints', 'model_final.pt'),
    os.path.join(PROJECT_ROOT, 'data', 'tokenizer.json'),
    device
)
ic = InferenceConfig()

for p in ['Привет', 'Как тебя зовут?', 'Пока']:
    context = f'{ic.system_prompt}\n\nUSER: {p}\nASSISTANT:'
    resp = generate_response(model, tokenizer, mc, context, device, temperature=0.8, max_new_tokens=50)
    print(f'User: {p}')
    print(f'AI: {resp}')
    print()

In [ ]:
# Скачать чекпоинт
from google.colab import files
files.download('checkpoints/model_final.pt')

In [ ]:
# Gradio интерфейс (опционально)
!pip install gradio -q
import subprocess, sys
proc = subprocess.Popen([sys.executable, 'app.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in iter(proc.stdout.readline, b''):
    print(line.decode(), end='')